# Module 6 — Loss Analysis

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path='/Users/toszmac/Documents/Pharma_Manufacturing_Data_Analysis_Project/Pharma_Manufacturing_Data_Analysis_Folder/.env')

engine = create_engine(
    f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

print("Connected to lli_db")

Connected to lli_db


## Q1 — What is the total material loss per stage in units?


In [2]:
material_loss_per_stage_query = """
    WITH stage_yields AS (
        SELECT
            f.jo_id,
            p.dosage_form,
            j.batch_size,
            MAX(CASE WHEN f.stage = 'dry_blending' THEN f.yield_pct END) AS yield_dry,
            MAX(CASE WHEN f.stage = 'compression' THEN f.yield_pct END) AS yield_cmprsn,
            MAX(CASE WHEN f.stage = 'coating' THEN f.yield_pct END) AS yield_ctng,
            MAX(CASE WHEN f.stage = 'encapsulation' THEN f.yield_pct END) AS yield_encap
        FROM fact_batch_production f
        JOIN dim_job_order j ON f.jo_id = j.jo_id
        JOIN dim_product p ON f.product_id = p.product_id
        WHERE f.yield_pct IS NOT NULL
        GROUP BY f.jo_id, p.dosage_form, j.batch_size
    ),
    stage_outputs AS (
        SELECT
            jo_id,
            dosage_form,
            batch_size,
            yield_dry,
            yield_cmprsn,
            yield_ctng,
            yield_encap,
            -- Stage outputs
            batch_size * COALESCE(yield_dry, 1)                         AS output_dry,
            batch_size * COALESCE(yield_dry, 1) * COALESCE(yield_cmprsn, 1) AS output_cmprsn,
            batch_size * COALESCE(yield_dry, 1) * COALESCE(yield_cmprsn, 1)
                       * COALESCE(yield_ctng, 1)                        AS output_ctng,
            batch_size * COALESCE(yield_dry, 1) * COALESCE(yield_encap, 1) AS output_encap
        FROM stage_yields
    ),
    stage_losses AS (
        SELECT
            jo_id,
            dosage_form,
            batch_size,
            -- Loss per stage = input to that stage × (1 - yield)
            CASE WHEN yield_dry IS NOT NULL
                THEN batch_size * (1 - yield_dry) END                   AS loss_dry,
            CASE WHEN yield_cmprsn IS NOT NULL
                THEN output_dry * (1 - yield_cmprsn) END                AS loss_cmprsn,
            CASE WHEN yield_ctng IS NOT NULL
                THEN output_cmprsn * (1 - yield_ctng) END               AS loss_ctng,
            CASE WHEN yield_encap IS NOT NULL
                THEN output_dry * (1 - yield_encap) END                 AS loss_encap
        FROM stage_outputs
    )
    SELECT
        stage,
        ROUND(SUM(loss_units)) AS total_loss_units
    FROM (
        SELECT 'dry_blending'  AS stage, loss_dry    AS loss_units FROM stage_losses
        UNION ALL
        SELECT 'compression'   AS stage, loss_cmprsn AS loss_units FROM stage_losses
        UNION ALL
        SELECT 'coating'       AS stage, loss_ctng   AS loss_units FROM stage_losses
        UNION ALL
        SELECT 'encapsulation' AS stage, loss_encap  AS loss_units FROM stage_losses
    ) unpivoted
    WHERE loss_units IS NOT NULL
    GROUP BY stage
    ORDER BY total_loss_units DESC;
"""

df_loss_per_stage = pd.read_sql_query(material_loss_per_stage_query, engine)
df_loss_per_stage

,stage,total_loss_units
0,coating,4951434.0
1,compression,3874382.0
2,dry_blending,2338837.0
3,encapsulation,1956866.0


In [3]:
stage_labels = {
    'dry_blending': 'Dry Blending',
    'compression': 'Compression',
    'coating': 'Coating',
    'encapsulation': 'Encapsulation'
}

df_loss_per_stage['stage_label'] = df_loss_per_stage['stage'].map(stage_labels)

fig = px.bar(
    df_loss_per_stage,
    x='stage_label',
    y='total_loss_units',
    title='Total Material Loss per Stage (2025)',
    labels={
        'total_loss_units': 'Total Loss (Units)',
        'stage_label': ''
    },
    text='total_loss_units',
    color='total_loss_units',
    color_continuous_scale='Reds'
)

fig.update_traces(
    textposition='outside',
    texttemplate='%{text:,.0f}'
)
fig.update_layout(
    plot_bgcolor='white',
    coloraxis_showscale=False,
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=False, title='Total Loss (Units)')
)
fig.show()

## Q1 — Total Material Loss per Stage
 
**Observation:**
Coating generates the highest absolute material loss at 4,951,434 units, followed by Compression at 3,874,382 units. Dry Blending accounts for 2,338,837 units and Encapsulation for 1,956,866 units. Total facility-wide material loss across all stages is approximately 13.1 million units in 2025.
 
**Insight:**
Coating's position as the top loss stage is counterintuitive given that its average yield (96.93%) is higher than Encapsulation (96.39%). The explanation lies in volume — coating processes the largest number of applicable batches and those batches tend to have larger batch sizes (Film-Coated Tablets dominate production). A small percentage loss on high-volume, large-batch products generates far more absolute unit loss than a larger percentage loss on low-volume products. This is the critical distinction between yield percentage and absolute loss.
 
**Recommendation:**
Present both yield percentage and absolute unit loss to management — they tell different stories and both are necessary for informed decision-making. A 3% yield gap on a 500,000-unit batch is 15,000 units lost. The same 3% gap on a 50,000-unit batch is 1,500 units. Prioritize loss reduction efforts on coating given its 4.95M unit loss, even though its yield percentage looks better than encapsulation.


## Q2 — What is the total material loss per dosage form?


In [4]:
material_loss_dosage_form_query = """
    WITH stage_yields AS (
        SELECT
            f.jo_id,
            p.dosage_form,
            j.batch_size,
            MAX(CASE WHEN f.stage = 'dry_blending' THEN f.yield_pct END) AS yield_dry,
            MAX(CASE WHEN f.stage = 'compression' THEN f.yield_pct END) AS yield_cmprsn,
            MAX(CASE WHEN f.stage = 'coating' THEN f.yield_pct END) AS yield_ctng,
            MAX(CASE WHEN f.stage = 'encapsulation' THEN f.yield_pct END) AS yield_encap
        FROM fact_batch_production f
        JOIN dim_job_order j ON f.jo_id = j.jo_id
        JOIN dim_product p ON f.product_id = p.product_id
        WHERE f.yield_pct IS NOT NULL
        GROUP BY f.jo_id, p.dosage_form, j.batch_size
    ),
    stage_outputs AS (
        SELECT
            jo_id,
            dosage_form,
            batch_size,
            batch_size * COALESCE(yield_dry, 1)                              AS output_dry,
            batch_size * COALESCE(yield_dry, 1) * COALESCE(yield_cmprsn, 1) AS output_cmprsn
        FROM stage_yields
    ),
    batch_losses AS (
        SELECT
            sy.dosage_form,
            COALESCE(sy.batch_size * (1 - sy.yield_dry), 0)             AS loss_dry,
            COALESCE(so.output_dry * (1 - sy.yield_cmprsn), 0)          AS loss_cmprsn,
            COALESCE(so.output_cmprsn * (1 - sy.yield_ctng), 0)         AS loss_ctng,
            COALESCE(so.output_dry * (1 - sy.yield_encap), 0)           AS loss_encap
        FROM stage_yields sy
        JOIN stage_outputs so ON sy.jo_id = so.jo_id
    )
    SELECT
        dosage_form,
        ROUND(SUM(loss_dry + loss_cmprsn + loss_ctng + loss_encap)) AS total_loss_units
    FROM batch_losses
    GROUP BY dosage_form
    ORDER BY total_loss_units DESC;
"""

df_loss_dosage_form = pd.read_sql_query(material_loss_dosage_form_query, engine)
df_loss_dosage_form

,dosage_form,total_loss_units
0,FILM-COATED TABLET,7134203.0
1,CAPSULE,2331287.0
2,SUSTAINED-RELEASE TABLET,2034654.0
3,TABLET,1284364.0
4,BILAYER TABLET,243175.0
5,EXTENDED RELEASE TABLET,41025.0
6,ENTERIC-COATED TABLET,38570.0
7,MODIFIED RELEASE TABLET,14242.0


In [5]:
fig = px.bar(
    df_loss_dosage_form,
    x='total_loss_units',
    y='dosage_form',
    orientation='h',
    title='Total Material Loss by Dosage Form (2025)',
    labels={
        'total_loss_units': 'Total Loss (Units)',
        'dosage_form': ''
    },
    text='total_loss_units',
    color='total_loss_units',
    color_continuous_scale='Reds'
)

fig.update_traces(
    textposition='auto',
    insidetextanchor='end',
    texttemplate='%{text:,.0f}'
)
fig.update_layout(
    plot_bgcolor='white',
    coloraxis_showscale=False,
    yaxis={'categoryorder': 'total ascending'},
    xaxis=dict(showgrid=False, title='Total Loss (Units)')
)
fig.show()

## Q2 — Total Material Loss by Dosage Form
 
**Observation:**
Film-Coated Tablet dominates with 7,134,203 units lost — more than 3x the next highest dosage form. Capsule (2,331,287) and Sustained-Release Tablet (2,034,654) are distant second and third. Tablet accounts for 1,284,364 units. Bilayer Tablet (243,175), Extended Release Tablet (41,025), Enteric-Coated Tablet (38,570), and Modified Release Tablet (14,242) contribute comparatively minor losses.
 
**Insight:**
Film-Coated Tablet's 7.1M unit loss is a direct consequence of its dominance in production volume — it represents 51.2% of total output. Even at a relatively good average yield, the sheer scale of Film-Coated Tablet production means small inefficiencies compound into massive absolute losses. Capsule and Sustained-Release Tablet losses are proportional to their production volumes and are less alarming in that context.
 
**Recommendation:**
Any yield improvement initiative targeting Film-Coated Tablets will have a disproportionately large impact on total facility loss. A 0.5% yield improvement on Film-Coated Tablet production alone would recover hundreds of thousands of units annually. This should be framed to management as a high-return, low-complexity improvement opportunity — focusing on KEVIN 48 coating consistency and compression tooling maintenance for film-coated products specifically.


## Q3 — Which generics have the highest total material loss?

In [6]:
material_loss_generic_query = """
    WITH stage_yields AS (
        SELECT
            f.jo_id,
            p.generic_name,
            p.dosage_form,
            j.batch_size,
            MAX(CASE WHEN f.stage = 'dry_blending' THEN f.yield_pct END) AS yield_dry,
            MAX(CASE WHEN f.stage = 'compression' THEN f.yield_pct END) AS yield_cmprsn,
            MAX(CASE WHEN f.stage = 'coating' THEN f.yield_pct END) AS yield_ctng,
            MAX(CASE WHEN f.stage = 'encapsulation' THEN f.yield_pct END) AS yield_encap
        FROM fact_batch_production f
        JOIN dim_job_order j ON f.jo_id = j.jo_id
        JOIN dim_product p ON f.product_id = p.product_id
        WHERE f.yield_pct IS NOT NULL
        GROUP BY f.jo_id, p.generic_name, p.dosage_form, j.batch_size
    ),
    stage_outputs AS (
        SELECT
            jo_id,
            generic_name,
            dosage_form,
            batch_size,
            batch_size * COALESCE(yield_dry, 1)                              AS output_dry,
            batch_size * COALESCE(yield_dry, 1) * COALESCE(yield_cmprsn, 1) AS output_cmprsn
        FROM stage_yields
    ),
    batch_losses AS (
        SELECT
            sy.generic_name,
            COALESCE(sy.batch_size * (1 - sy.yield_dry), 0)             AS loss_dry,
            COALESCE(so.output_dry * (1 - sy.yield_cmprsn), 0)          AS loss_cmprsn,
            COALESCE(so.output_cmprsn * (1 - sy.yield_ctng), 0)         AS loss_ctng,
            COALESCE(so.output_dry * (1 - sy.yield_encap), 0)           AS loss_encap
        FROM stage_yields sy
        JOIN stage_outputs so ON sy.jo_id = so.jo_id
    )
    SELECT
        generic_name,
        ROUND(SUM(loss_dry + loss_cmprsn + loss_ctng + loss_encap)) AS total_loss_units
    FROM batch_losses
    GROUP BY generic_name
    ORDER BY total_loss_units DESC
    LIMIT 15;
"""

df_loss_generic = pd.read_sql_query(material_loss_generic_query, engine)

df_loss_generic['generic_short'] = df_loss_generic['generic_name'].apply(
    lambda x: x[:40] + '...' if len(x) > 40 else x
)

df_loss_generic

,generic_name,total_loss_units,generic_short
0,ATORVASTATIN CALCIUM,2022231.0,ATORVASTATIN CALCIUM
1,BUTAMIRATE CITRATE,1341002.0,BUTAMIRATE CITRATE
2,MEFENAMIC ACID,1056198.0,MEFENAMIC ACID
3,PARACETAMOL + THIAMINE MONONITRATE (VITAMIN B1...,662266.0,PARACETAMOL + THIAMINE MONONITRATE (VITA...
4,ASCORBIC ACID (AS SODIUM ASCORBATE) + ZINC (AS...,649629.0,ASCORBIC ACID (AS SODIUM ASCORBATE) + ZI...
5,ISOSORBIDE MONONITRATE,582609.0,ISOSORBIDE MONONITRATE
6,SITAGLIPTIN (AS PHOSPHATE MONOHYDRATE) + METFO...,548119.0,SITAGLIPTIN (AS PHOSPHATE MONOHYDRATE) +...
7,ASCORBIC ACID (SODIUM ASCORBATE) + ZINC (AS GL...,520547.0,ASCORBIC ACID (SODIUM ASCORBATE) + ZINC ...
8,DAPAGLIFLOZIN,428064.0,DAPAGLIFLOZIN
9,METFORMIN HYDROCHLORIDE,410609.0,METFORMIN HYDROCHLORIDE


In [7]:
fig = px.bar(
    df_loss_generic,
    x='total_loss_units',
    y='generic_short',
    orientation='h',
    title='Top 15 Generics by Total Material Loss (2025)',
    labels={
        'total_loss_units': 'Total Loss (Units)',
        'generic_short': ''
    },
    text='total_loss_units',
    color='total_loss_units',
    color_continuous_scale='Reds'
)

fig.update_traces(
    textposition='auto',
    insidetextanchor='end',
    texttemplate='%{text:,.0f}'
)
fig.update_layout(
    plot_bgcolor='white',
    coloraxis_showscale=False,
    yaxis={'categoryorder': 'total ascending'},
    xaxis=dict(showgrid=False, title='Total Loss (Units)')
)
fig.show()

## Q3 — Top 15 Generics by Total Material Loss
 
**Observation:**
Atorvastatin Calcium leads by a significant margin at 2,022,231 units lost, followed by Butamirate Citrate at 1,341,002 and Mefenamic Acid at 1,056,198. These three generics collectively account for approximately 4.4 million units of loss — roughly 34% of the total facility loss of 13.1 million units. The remaining 12 generics in the top 15 range from 312,979 to 662,266 units.
 
**Insight:**
Atorvastatin Calcium's top position is notable because it was also identified in Module 2 as a product with naming inconsistencies that required data cleaning — confirming it is a high-volume, high-frequency generic that warrants close operational attention. Butamirate Citrate leads in batch count (239 batches from Module 2), so its high loss is volume-driven. Mefenamic Acid's 1,056,198 unit loss relative to its batch count should be investigated — if it has fewer batches than Butamirate Citrate but comparable loss, its per-batch loss rate is higher, suggesting a yield efficiency problem rather than a volume problem.
 
**Recommendation:**
Conduct a per-batch loss rate analysis for the top 3 generics to separate volume-driven loss from efficiency-driven loss. For Atorvastatin Calcium and Butamirate Citrate, loss reduction should focus on process consistency at scale. For Mefenamic Acid, investigate whether a specific formulation characteristic — particle size, compressibility, coating adhesion — is driving higher-than-expected per-batch losses. These three generics should be the first candidates for a formal yield improvement project in 2026.
 


## Q4 — What is the monthly trend of material loss?

In [8]:
material_loss_monthly_query = """
    WITH stage_yields AS (
        SELECT
            f.jo_id,
            j.batch_size,
            d.year,
            TO_CHAR(d.full_date, 'YYYY-MM') AS month,
            TO_CHAR(d.full_date, 'Mon') AS month_short,
            MAX(CASE WHEN f.stage = 'dry_blending' THEN f.yield_pct END) AS yield_dry,
            MAX(CASE WHEN f.stage = 'compression' THEN f.yield_pct END) AS yield_cmprsn,
            MAX(CASE WHEN f.stage = 'coating' THEN f.yield_pct END) AS yield_ctng,
            MAX(CASE WHEN f.stage = 'encapsulation' THEN f.yield_pct END) AS yield_encap
        FROM fact_batch_production f
        JOIN dim_job_order j ON f.jo_id = j.jo_id
        JOIN dim_date d ON f.date_id = d.date_id
        WHERE f.yield_pct IS NOT NULL
        AND d.year = 2025
        AND f.stage = 'dry_blending'
        GROUP BY f.jo_id, j.batch_size, d.year,
                 TO_CHAR(d.full_date, 'YYYY-MM'),
                 TO_CHAR(d.full_date, 'Mon')
    ),
    stage_outputs AS (
        SELECT
            jo_id,
            month,
            month_short,
            batch_size,
            batch_size * COALESCE(yield_dry, 1)                              AS output_dry,
            batch_size * COALESCE(yield_dry, 1) * COALESCE(yield_cmprsn, 1) AS output_cmprsn,
            yield_dry,
            yield_cmprsn,
            yield_ctng,
            yield_encap
        FROM stage_yields
    )
    SELECT
        month,
        month_short,
        ROUND(SUM(
            COALESCE(batch_size * (1 - yield_dry), 0) +
            COALESCE(output_dry * (1 - yield_cmprsn), 0) +
            COALESCE(output_cmprsn * (1 - yield_ctng), 0) +
            COALESCE(output_dry * (1 - yield_encap), 0)
        )) AS total_loss_units
    FROM stage_outputs
    GROUP BY month, month_short
    ORDER BY month;
"""

df_loss_monthly = pd.read_sql_query(material_loss_monthly_query, engine)
df_loss_monthly

,month,month_short,total_loss_units
0,2025-01,Jan,226741.0
1,2025-02,Feb,306010.0
2,2025-03,Mar,151711.0
3,2025-04,Apr,159760.0
4,2025-05,May,212220.0
5,2025-06,Jun,107914.0
6,2025-07,Jul,208487.0
7,2025-08,Aug,204927.0
8,2025-09,Sep,369190.0
9,2025-10,Oct,155657.0


In [9]:
fig = px.line(
    df_loss_monthly,
    x='month_short',
    y='total_loss_units',
    title='Monthly Material Loss Trend (2025)',
    labels={
        'total_loss_units': 'Total Loss (Units)',
        'month_short': ''
    },
    text='total_loss_units',
    markers=True
)

fig.update_traces(
    line=dict(color='crimson', width=2),
    marker=dict(size=8, color='crimson'),
    textposition='top center',
    texttemplate='%{text:,.0f}'
)
fig.update_layout(
    plot_bgcolor='white',
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=True, gridcolor='lightgrey', title='Total Loss (Units)')
)
fig.show()

## Q4 — Monthly Material Loss Trend
 
**Observation:**
September is the peak loss month at 369,190 units — the highest point in the year by a significant margin. February is the second highest at 306,010 units. June is the lowest at 107,914 units, followed by December at 49,454 units. The overall pattern shows two loss peaks (February and September) separated by a mid-year trough, with losses declining sharply in the final two months of the year.
 
**Insight:**
The September peak aligns directly with your Q3 production peak — August had the highest batch count (120 batches) and September processes the tail end of that surge. High production volume compresses scheduling, increases machine utilization pressure, and reduces time for setup optimization — all of which contribute to yield degradation and higher absolute loss. The December low reflects the same dosage form mix shift observed in the lead time analysis — simpler products with inherently lower loss rates being prioritized at year-end.
 
**Recommendation:**
Use the September loss peak as the anchor point for a Q3 preparation protocol. If the facility anticipates another Q3 surge in 2026, pre-schedule preventive maintenance for KEVIN 48, PHARMAFILL, and ZP-41 before August to ensure machines enter the high-volume period in optimal condition. Additionally, review the product scheduling mix for September specifically — if high-loss products like Atorvastatin Calcium and Mefenamic Acid are concentrated in that month, redistributing some of their batches to lower-volume months could meaningfully reduce the September loss peak.
